# FireCrawl

## Web search

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

FIRECRAWL_API = os.environ["FIRECRAWL_API_KEY"]

url = "https://api.firecrawl.dev/v2/search"

payload = {
    "query": "what are the latest news about Jensen Huang?",
    "sources": ["web"],
    "categories": [],
    "limit": 3,
    "scrapeOptions": {
        "onlyMainContent": True,
        "maxAge": 172800000,
        "parsers": ["pdf"],
        "formats": [],
    },
}

headers = {
    "Authorization": f"Bearer {FIRECRAWL_API}",
    "Content-Type": "application/json",
}

response = requests.post(url, json=payload, headers=headers)

print(response.json())


In [9]:
response.json().keys()

dict_keys(['success', 'data', 'creditsUsed', 'id'])

In [11]:
response.json()["creditsUsed"]

2

In [13]:
response.json()["data"].keys()

dict_keys(['web'])

In [15]:
search_results = response.json()["data"]["web"]
len(search_results)

3

In [18]:
search_results[0].keys()

dict_keys(['url', 'title', 'description', 'position'])

In [19]:
search_results[0]["url"]

'https://www.cbsnews.com/news/jensen-huang-nvidia-ceo-return-statement/'

In [20]:
search_results[0]["title"]

'Jensen Huang breaks silence after resigning from Nvidia, says he " ...'

In [24]:
search_results[0]["position"]

1

In [36]:
import json 
with open("jensen_huang.json", "w") as f:
    json.dump(search_results, f, ensure_ascii=False, indent=4)

In [ ]:
import json
with open("jensen_huang.json",encoding="utf-8") as f:
    data = json.load(f)

data

## Webscraping

- scrape specific page

In [ ]:
from firecrawl import Firecrawl

app = Firecrawl()

data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evals/",
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"],
)

In [41]:
data.__dict__.keys()

dict_keys(['agent_hints', 'markdown', 'html', 'raw_html', 'json', 'summary', 'metadata', 'links', 'images', 'screenshot', 'audio', 'video', 'actions', 'answer', 'highlights', 'warning', 'change_tracking', 'branding', 'product', 'menu', 'pages', 'blocks', 'tools'])

In [52]:
metadata = data.metadata
metadata.__dict__.keys()

dict_keys(['title', 'description', 'url', 'language', 'keywords', 'robots', 'og_title', 'og_description', 'og_url', 'og_image', 'og_audio', 'og_determiner', 'og_locale', 'og_locale_alternate', 'og_site_name', 'og_video', 'favicon', 'dc_terms_created', 'dc_date_created', 'dc_date', 'dc_terms_type', 'dc_type', 'dc_terms_audience', 'dc_terms_subject', 'dc_subject', 'dc_description', 'dc_terms_keywords', 'modified_time', 'published_time', 'article_tag', 'article_section', 'source_url', 'status_code', 'scrape_id', 'num_pages', 'total_pages', 'content_type', 'proxy_used', 'timezone', 'cache_state', 'cached_at', 'credits_used', 'concurrency_limited', 'concurrency_queue_duration_ms', 'error'])

In [60]:
print(f"{metadata.title = }")
print(f"{metadata.description = }")
print(f"{metadata.url = }")
print(f"{metadata.language = }")
print(f"{metadata.keywords = }")
print(f"{metadata.robots = }")
print(f"{metadata.proxy_used = }")

metadata.title = 'Pydantic Evals | Pydantic Docs'
metadata.description = 'Pydantic Evals is a code-first Python framework for evaluating LLM apps and agents: score outputs and tool-call trajectories on datasets or live traffic.'
metadata.url = 'https://pydantic.dev/docs/ai/evals/evals/'
metadata.language = 'en'
metadata.keywords = None
metadata.robots = None
metadata.proxy_used = 'basic'


In [63]:
metadata.title

'Pydantic Evals | Pydantic Docs'

In [ ]:
with open(f"{metadata.title.replace('|', '')}.md", "w", encoding="utf-8") as f:
    f.write(data.markdown)

## Crawling

In [66]:
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions

app = Firecrawl()

scrape_opts = ScrapeOptions(
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"]
)

crawl_result = app.crawl(
    "https://www.myh.se/validering-och-seqf",
    sitemap="include",
    crawl_entire_domain=False,
    limit=10,
    scrape_options=scrape_opts
)


In [68]:
crawl_result.__dict__.keys()

dict_keys(['id', 'status', 'total', 'completed', 'credits_used', 'expires_at', 'next', 'warning', 'data'])

In [70]:
crawl_result.credits_used

10

In [73]:
from pathlib import Path

Path("myh").mkdir(exist_ok=True)

for document in crawl_result.data:
    metadata = document.metadata
    with open(f"myh/{metadata.title.replace('|', '')}.md", "w", encoding="utf-8") as f:
        f.write(document.markdown)